# Life Expectancy By Country — Solution

**Goal:** Investigate how a country’s economic success (GDP) relates to average life expectancy across 158 countries. Full working solutions, alternate implementations, extra practice, and a parameterised simulation.

**Data:** `data/country_data.csv`

**Audience notes:** Same as the skeleton — adapt depth and language for data-literate analysts, subject-matter experts, and time-pressed executives / nonspecialists.

---

## Flowchart of the Desired Outcome

![Life Expectancy Analysis Flowchart](life_expectancy_flowchart.png)

## Step 0 — Setup & Load Packages

In [ ]:
# load packages
library(ggplot2)
library(readr)
library(dplyr)

## Step 1 — Access the Data

Inspect the first rows and column names.

In [ ]:
# import and inspect data
data <- read_csv("data/country_data.csv")
head(data)
names(data)
glimpse(data)   # or str(data)

## Step 2 — Isolate the Life Expectancy Column as a Vector

In [ ]:
# life expectancy vector (dplyr::pull returns a plain vector)
life_expectancy <- data %>%
  pull(life_expectancy)

# quick check
head(life_expectancy)
length(life_expectancy)
class(life_expectancy)

## Step 3 — Find the Quartiles

In [ ]:
# life expectancy quartiles (0%, 25%, 50%, 75%, 100%)
life_expectancy_quartiles <- quantile(life_expectancy)
life_expectancy_quartiles

# also useful:
# quantile(life_expectancy, probs = c(0.25, 0.5, 0.75))

## Step 4 — Plot the Histogram of Life Expectancy

In [ ]:
# plot histogram of life expectancy
hist(life_expectancy,
     main = "Distribution of Life Expectancy (158 countries)",
     xlab = "Life Expectancy (years)",
     col = "steelblue",
     border = "white")
abline(v = life_expectancy_quartiles[2:4], col = c("orange", "red", "darkgreen"), lwd = 2, lty = 2)
legend("topleft", legend = c("Q1", "Median", "Q3"),
       col = c("orange", "red", "darkgreen"), lty = 2, lwd = 2, bty = "n")

## Step 5 — Interpret a Life Expectancy of 70 Years

From the quartiles printed above, locate where 70 falls.

- If 70 is between the 25 % and 50 % quantiles → second quarter.
- If between the 50 % and 75 % quantiles → third quarter.
- etc.

(In the synthetic data used here the median is typically near 71, so 70 often sits in the second or third quarter depending on the exact draw.)

## Step 6 — Isolate the GDP Column

In [ ]:
# gdp vector
gdp <- data %>%
  pull(GDP)

head(gdp)
summary(gdp)

## Step 7 — Find the Median GDP

In [ ]:
# median gdp (two equivalent ways)
median_gdp <- median(gdp)
# median_gdp <- quantile(gdp, 0.5)

median_gdp

## Step 8 — Split into Low-GDP and High-GDP Life-Expectancy Vectors

In [ ]:
# low gdp life expectancy vector
low_gdp <- data %>%
  filter(GDP <= median_gdp) %>%
  pull(life_expectancy)

# high gdp life expectancy vector
high_gdp <- data %>%
  filter(GDP > median_gdp) %>%
  pull(life_expectancy)

length(low_gdp)
length(high_gdp)

## Step 9 — Quartiles of the Low-GDP Group

In [ ]:
# low gdp quartiles
low_gdp_quartiles <- quantile(low_gdp)
low_gdp_quartiles

## Step 10 — Quartiles of the High-GDP Group

In [ ]:
# high gdp quartiles
high_gdp_quartiles <- quantile(high_gdp)
high_gdp_quartiles

## Step 11 — Histograms of the Two Groups

In [ ]:
# plot low gdp histogram
hist(low_gdp,
     col = "red",
     main = "Life Expectancy — Low GDP Countries",
     xlab = "Life Expectancy (years)",
     xlim = range(c(low_gdp, high_gdp)))

# plot high gdp histogram
hist(high_gdp,
     col = "blue",
     main = "Life Expectancy — High GDP Countries",
     xlab = "Life Expectancy (years)",
     xlim = range(c(low_gdp, high_gdp)))

## Step 12 — Interpret 70 Years in Each Group

Compare 70 against `low_gdp_quartiles` and `high_gdp_quartiles`.

Typical pattern (observed with this data):

- In the **high-GDP** group the whole distribution is shifted right; 70 years may fall in the lower half (1st or 2nd quarter).
- In the **low-GDP** group 70 years is often in the upper half (3rd or 4th quarter).

This is strong visual evidence that higher national wealth is associated with longer average life expectancy.

## Alternate Code Paths

In [ ]:
# ALTERNATE 1 — base-R extraction + fivenum / summary
life_expectancy_base <- data$life_expectancy          # or data[["life_expectancy"]]
fivenum(life_expectancy_base)                         # Tukey five-number summary
summary(life_expectancy_base)

# ALTERNATE 2 — logical indexing instead of filter()
low_gdp_alt  <- life_expectancy[gdp <= median_gdp]
high_gdp_alt <- life_expectancy[gdp >  median_gdp]
quantile(low_gdp_alt)
quantile(high_gdp_alt)

# ALTERNATE 3 — create a factor and use tapply / split
gdp_group <- factor(ifelse(gdp <= median_gdp, "Low", "High"),
                    levels = c("Low", "High"))
tapply(life_expectancy, gdp_group, quantile)
# or
split_le <- split(life_expectancy, gdp_group)
lapply(split_le, quantile)

# ALTERNATE 4 — ggplot2 histograms (faceted)
library(ggplot2)
plot_df <- data.frame(
  le = c(low_gdp, high_gdp),
  group = rep(c("Low GDP", "High GDP"), c(length(low_gdp), length(high_gdp)))
)
ggplot(plot_df, aes(x = le, fill = group)) +
  geom_histogram(bins = 15, colour = "white", alpha = 0.8) +
  facet_wrap(~ group, ncol = 1) +
  scale_fill_manual(values = c("Low GDP" = "#e74c3c", "High GDP" = "#3498db")) +
  labs(title = "Life Expectancy by GDP Group",
       x = "Life Expectancy (years)", y = "Count") +
  theme_minimal() +
  theme(legend.position = "none")

## More Practice — Solutions

In [ ]:
# 1. IQR of overall and of each group
iqr_overall <- IQR(life_expectancy)
iqr_low     <- IQR(low_gdp)
iqr_high    <- IQR(high_gdp)
cat("IQR overall:", iqr_overall, "\n")
cat("IQR low GDP:", iqr_low, "\n")
cat("IQR high GDP:", iqr_high, "\n")

# 2. Mean vs median (skewness indicator)
cat("\nLow GDP  — mean:", mean(low_gdp), " median:", median(low_gdp), "\n")
cat("High GDP — mean:", mean(high_gdp), " median:", median(high_gdp), "\n")

# 3. Middle 50 % of GDP
q25_gdp <- quantile(gdp, 0.25)
q75_gdp <- quantile(gdp, 0.75)
mid_gdp <- data %>%
  filter(GDP > q25_gdp, GDP <= q75_gdp) %>%
  pull(life_expectancy)
cat("\nMiddle-GDP group size:", length(mid_gdp), "\n")
quantile(mid_gdp)

# 4. Quick correlation (numeric association)
cor(life_expectancy, gdp, method = "spearman")

## Simulation Section — Parameterised Experiment

Change `split_prob`, `noise_sd` or `n_sim` and re-run the cell to see how the gap between low- and high-GDP life expectancy changes.

In [ ]:
# SIMULATION parameters (edit these)
set.seed(42)
split_prob <- 0.5      # GDP quantile used as threshold (0.5 = median)
noise_sd   <- 1.5      # SD of Gaussian noise added to life expectancy
n_sim      <- 300      # number of Monte-Carlo replicates

# Observed difference under the chosen split (with optional noise)
le_noisy <- life_expectancy + rnorm(length(life_expectancy), 0, noise_sd)
threshold <- quantile(gdp, split_prob)
low_sim  <- le_noisy[gdp <= threshold]
high_sim <- le_noisy[gdp >  threshold]
obs_diff <- median(high_sim) - median(low_sim)
cat("Observed median difference (High - Low) under current params:",
    round(obs_diff, 2), "years\n")

# Monte-Carlo distribution of the difference
diffs <- replicate(n_sim, {
  le_n <- life_expectancy + rnorm(length(life_expectancy), 0, noise_sd)
  thr  <- quantile(gdp, split_prob)
  median(le_n[gdp > thr]) - median(le_n[gdp <= thr])
})

hist(diffs,
     main = paste0("Monte-Carlo: High-Low median LE difference\n",
                   "(split_prob = ", split_prob, ", noise_sd = ", noise_sd, ")"),
     xlab = "Median LE (High GDP) - Median LE (Low GDP)",
     col = "purple", border = "white")
abline(v = mean(diffs), col = "red", lwd = 2)
abline(v = quantile(diffs, c(0.025, 0.975)), col = "orange", lty = 2)
legend("topleft",
       legend = c(paste("Mean diff =", round(mean(diffs), 2)),
                  "95% Monte-Carlo interval"),
       col = c("red", "orange"), lty = c(1, 2), lwd = c(2, 1), bty = "n")

cat("Mean simulated difference:", round(mean(diffs), 2), "\n")
cat("95% MC interval: [", round(quantile(diffs, 0.025), 2), ",",
    round(quantile(diffs, 0.975), 2), "]\n")

## Audience-Adapted Takeaways

**For a data analyst / technical colleague**  
Life expectancy is right-skewed overall. Splitting at the median GDP produces two clearly separated distributions: the high-GDP group has a higher median and a compressed upper tail, while the low-GDP group shows greater left-tail mass (countries with very low LE). Spearman correlation between GDP and LE is positive and moderately strong. Next steps could include a simple linear or log-linear model, controls for region, and formal tests of stochastic dominance.

**For a busy executive / policy maker**  
**Headline:** Countries above the median GDP live, on average, several years longer than countries below it.  
The visual comparison of the two histograms makes the gap obvious even without statistics training.  
**Implication:** Economic development and health outcomes move together; investments that raise national income are associated with longer lives, although causation cannot be claimed from this cross-section alone.